<a href="https://colab.research.google.com/github/jayvishalshah/-InfinityLoopRT/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import userdata

try:
    token = userdata.get('HF_TOKEN')
    print("✅ HF_TOKEN loaded successfully!")
    print("Token character length:", len(token))
except Exception as e:
    print("❌ Could not load HF_TOKEN. Please check your Colab Secrets setup.")
    print("Error details:", e)

✅ HF_TOKEN loaded successfully!
Token character length: 37


In [3]:
import os

# Create the 'core' folder first
os.makedirs('core', exist_ok=True)
print("✅ 'core' folder created (or already exists).")

✅ 'core' folder created (or already exists).


In [4]:
%%writefile core/interfaces.py
from abc import ABC, abstractmethod
from typing import Any, Dict, List, Optional

class Target(ABC):
    """Interface for the system being red-teamed (e.g., Gemini)."""

    @abstractmethod
    def send_prompt(self, prompt: str) -> str:
        """Sends a prompt to the target and returns the text response."""
        pass

class Brain(ABC):
    """Interface for the attacker/red-team model (e.g., Dolphin 3)."""

    @abstractmethod
    def generate_attack(self, target_info: str, context: Dict[str, Any]) -> str:
        """Generates the next prompt to test against the target."""
        pass

class Evaluator(ABC):
    """Interface for evaluating if an attack succeeded or failed."""

    @abstractmethod
    def evaluate(self, prompt: str, response: str) -> Dict[str, Any]:
        """Analyzes the response and returns a score/flag dictionary."""
        pass

class Memory(ABC):
    """Interface for tracking the history of the red-team session."""

    @abstractmethod
    def add_interaction(self, prompt: str, response: str, evaluation: Dict[str, Any]) -> None:
        """Stores a single turn of the conversation."""
        pass

    @abstractmethod
    def get_context(self) -> Dict[str, Any]:
        """Retrieves the history to feed back into the Brain."""
        pass

Writing core/interfaces.py


In [5]:
%%writefile core/config.py
import os
from dataclasses import dataclass
from typing import Optional

@dataclass
class ILRTConfig:
    """Global configuration for the Infinity Loop Red Teaming framework."""

    # Target Model Settings (The system being tested)
    target_api_key: Optional[str] = None
    target_model_name: str = "gemini-1.5-pro-latest"

    # Brain Model Settings (The attacker)
    hf_token: Optional[str] = None
    brain_model_name: str = "cognitivecomputations/dolphin-2.9-llama3-8b"

    # Loop & Strategy Settings
    max_turns: int = 5
    temperature: float = 0.7

    # Storage Paths
    output_dir: str = "outputs"

    def __post_init__(self):
        """Automatically create the output directory if it doesn't exist."""
        os.makedirs(self.output_dir, exist_ok=True)

Writing core/config.py
